# 🔋 NASA Battery EDA — Google Colab Version
> **Project:** `nasa-battery-eda` | CIT Kokrajhar PhD Research  
> **Dataset:** NASA PCoE Battery Dataset — B0005, B0006, B0007, B0018

---
## 📋 Before you run this notebook:

**Step 1:** Download the dataset from NASA:  
👉 https://phm-datasets.s3.amazonaws.com/NASA/5.+Battery+Data+Set.zip

**Step 2:** Upload the 4 files to your Google Drive in this folder:
```
My Drive/
  └── nasa_battery_data/
        ├── B0005.mat
        ├── B0006.mat
        ├── B0007.mat
        └── B0018.mat
```

**Step 3:** Run each cell one by one (Shift + Enter)

---

## CELL 1 — Mount Google Drive
This connects Colab to your Google Drive. A popup will ask you to sign in — allow it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

## CELL 2 — Set the path to your data
This tells the notebook where your `.mat` files are. Change the folder name if you named it differently.

In [ ]:
import os

# ✏️  Change this if you used a different folder name in Drive
DATA_FOLDER = '/content/drive/MyDrive/nasa_battery_data'

# Check the files are there
print('Files found in your data folder:')
for f in sorted(os.listdir(DATA_FOLDER)):
    print(f'  ✅ {f}')

## CELL 3 — Install & import libraries
Most libraries are already in Colab. We just need to import them.

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.15)
plt.rcParams['figure.dpi'] = 120

BATTERY_IDS        = ['B0005', 'B0006', 'B0007', 'B0018']
RATED_CAPACITY_AH  = 1.86
EOL_THRESHOLD_SOH  = 0.70

print('✅ All libraries imported!')

## CELL 4 — Understand the raw data structure
NASA stores data in MATLAB format (`.mat`). Let's look inside B0005 to understand the layout.

In [ ]:
# Load B0005 and inspect its structure
mat = sio.loadmat(f'{DATA_FOLDER}/B0005.mat', simplify_cells=True)

b      = mat['B0005']
cycles = b['cycle']

print(f'Total cycles in B0005: {len(cycles)}')
print(f'Fields in each cycle : {list(cycles[0].keys())}')

# Count cycle types
types = [c['type'] for c in cycles]
unique, counts = np.unique(types, return_counts=True)
print('\nCycle type breakdown:')
for t, n in zip(unique, counts):
    print(f'  {t:<12}: {n} cycles')

## CELL 5 — Load all 4 batteries

In [ ]:
def load_battery(mat_path, battery_id):
    """Load a NASA .mat file and return parsed cycles."""
    mat    = sio.loadmat(mat_path, simplify_cells=True)
    cycles_raw = mat[battery_id]['cycle']
    cycles = []
    for i, c in enumerate(cycles_raw):
        data = {}
        for field, val in c.get('data', {}).items():
            data[field] = np.atleast_1d(val).flatten()
        cycles.append({
            'cycle_index':    i + 1,
            'type':           str(c.get('type', '')).strip(),
            'ambient_temp_C': float(c.get('ambient_temperature', np.nan)),
            'data':           data,
        })
    return {'id': battery_id, 'cycles': cycles}


batteries = {}
for bid in BATTERY_IDS:
    path = f'{DATA_FOLDER}/{bid}.mat'
    batteries[bid] = load_battery(path, bid)
    disc = sum(1 for c in batteries[bid]['cycles'] if c['type'] == 'discharge')
    print(f'✅ {bid}: {len(batteries[bid]["cycles"])} total cycles | {disc} discharge')

## CELL 6 — Extract discharge capacity & compute SOH

**SOH = Capacity measured / 1.86 Ah**  
Capacity = integral of current over time during each discharge

In [ ]:
rows = []
for bid, bat in batteries.items():
    disc_num = 0
    for c in bat['cycles']:
        if c['type'] != 'discharge':
            continue
        d = c['data']
        if 'Voltage_measured' not in d:
            continue
        disc_num += 1
        voltage = d['Voltage_measured']
        current = d['Current_measured']
        temp    = d['Temperature_measured']
        time_s  = d.get('Time', np.arange(len(voltage)))

        capacity_Ah = float(np.trapz(np.abs(current), time_s) / 3600)
        soh         = min(capacity_Ah / RATED_CAPACITY_AH, 1.0)

        rows.append({
            'battery_id':      bid,
            'discharge_cycle': disc_num,
            'ambient_temp_C':  c['ambient_temp_C'],
            'capacity_Ah':     round(capacity_Ah, 5),
            'soh':             round(soh, 5),
            'voltage_max_V':   float(np.nanmax(voltage)),
            'voltage_min_V':   float(np.nanmin(voltage)),
            'temp_max_C':      float(np.nanmax(temp)),
        })

df = pd.DataFrame(rows)
print(f'Total rows: {len(df)}')
print(df.groupby('battery_id')[['capacity_Ah','soh']].agg(['min','max']).round(3))

## CELL 7 — Save results to Drive

In [ ]:
save_path = f'{DATA_FOLDER}/discharge_summary.csv'
df.to_csv(save_path, index=False)
print(f'✅ Saved → {save_path}')
df.head()

## CELL 8 — Plot: Capacity Fade & SOH Curves 📊
The key diagnostic plot — suitable for your paper.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Capacity (Ah)
ax = axes[0]
for bid, grp in df.groupby('battery_id'):
    ax.plot(grp['discharge_cycle'], grp['capacity_Ah'], lw=1.8, label=bid)
ax.axhline(RATED_CAPACITY_AH * EOL_THRESHOLD_SOH, color='red', ls='--', lw=1.5,
           label=f'EOL threshold (70% SOH)')
ax.set_title('Discharge Capacity Fade', fontweight='bold')
ax.set_xlabel('Discharge Cycle')
ax.set_ylabel('Capacity (Ah)')
ax.legend()

# Right: SOH (%)
ax = axes[1]
for bid, grp in df.groupby('battery_id'):
    ax.plot(grp['discharge_cycle'], grp['soh'] * 100, lw=1.8, label=bid)
ax.axhline(EOL_THRESHOLD_SOH * 100, color='red', ls='--', lw=1.5, label='EOL (70%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('State-of-Health (SOH) over Lifetime', fontweight='bold')
ax.set_xlabel('Discharge Cycle')
ax.set_ylabel('SOH (%)')
ax.legend()

plt.suptitle('NASA PCoE Battery Dataset — B0005 to B0018',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{DATA_FOLDER}/capacity_fade_soh.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure saved to your Drive!')

## CELL 9 — Plot: V / I / T signals for one discharge cycle

In [ ]:
bat   = batteries['B0005']
first = next(c for c in bat['cycles'] if c['type'] == 'discharge')
d     = first['data']

time_s  = d.get('Time', np.arange(len(d['Voltage_measured'])))
voltage = d['Voltage_measured']
current = d['Current_measured']
temp    = d['Temperature_measured']

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(time_s, voltage, color='steelblue', lw=1.5)
axes[0].set_ylabel('Voltage (V)')
axes[0].set_title('B0005 — First Discharge Cycle Signals', fontweight='bold')

axes[1].plot(time_s, current, color='darkorange', lw=1.5)
axes[1].set_ylabel('Current (A)')

axes[2].plot(time_s, temp, color='seagreen', lw=1.5)
axes[2].set_ylabel('Temperature (°C)')
axes[2].set_xlabel('Time (s)')

for ax in axes:
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(f'{DATA_FOLDER}/first_discharge_signals.png', dpi=150)
plt.show()
print('✅ Figure saved!')

## CELL 10 — EOL Statistics Table (for your paper)
When does each battery reach End-of-Life (SOH < 70%)?

In [ ]:
eol_stats = []
for bid, grp in df.groupby('battery_id'):
    below_eol = grp[grp['soh'] < EOL_THRESHOLD_SOH]
    eol_cycle = int(below_eol['discharge_cycle'].min()) if not below_eol.empty else 'Not reached'
    eol_stats.append({
        'Battery':                bid,
        'Total Discharge Cycles': len(grp),
        'Initial Capacity (Ah)':  grp['capacity_Ah'].iloc[0],
        'Final Capacity (Ah)':    grp['capacity_Ah'].iloc[-1],
        'Initial SOH (%)':        f"{grp['soh'].iloc[0]*100:.1f}",
        'Final SOH (%)':          f"{grp['soh'].iloc[-1]*100:.1f}",
        'EOL Cycle (SOH<70%)':    eol_cycle,
    })

df_eol = pd.DataFrame(eol_stats).set_index('Battery')
print('\n📊 EOL Statistics Table (suitable for Paper 1/2)\n')
df_eol

---
## ✅ You're done with Project 1!

### What you produced:
- `discharge_summary.csv` — clean data for all 4 batteries (saved to Drive)
- `capacity_fade_soh.png` — paper-ready degradation plot
- `first_discharge_signals.png` — V/I/T signal plot
- EOL statistics table

### ➡️ Next step: Project 2 — Feature Engineering
We will extract 29 health indicators from this data to feed into the QUBO selection pipeline.
